# 04 – Load into SQLite

Load the cleaned payments into a SQLite database so the analysis can be done in SQL. Dates are stored as text (`YYYY-MM-DD`) and flags as 0/1, because SQLite has no separate date or true/false types.

In [3]:
import pandas as pd
import sqlite3
from pathlib import Path

df = pd.read_csv(
    "../data/processed/spend_clean.csv",
    dtype={"transaction_number": str},
    parse_dates=["pay_date", "report_month"],
)

# SQLite-friendly formats
df["pay_date"] = df["pay_date"].dt.strftime("%Y-%m-%d")
df["report_month"] = df["report_month"].dt.strftime("%Y-%m")
for c in ["date_mismatch", "is_credit", "is_zero", "is_under_500"]:
    df[c] = df[c].astype(int)

# A unique ID per row, so individual payments can be referred to later
df.insert(0, "payment_id", range(1, len(df) + 1))

DB = Path("../data/processed/procurement.db")
conn = sqlite3.connect(DB)
df.to_sql("payments", conn, if_exists="replace", index=False)
conn.execute("CREATE INDEX IF NOT EXISTS idx_supplier ON payments(supplier_clean)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_month ON payments(report_month)")
conn.commit()

print(pd.read_sql("SELECT COUNT(*) AS payments, ROUND(SUM(amount), 2) AS total FROM payments", conn))
print()
print(pd.read_sql("""
    SELECT spend_group, COUNT(*) AS payments, ROUND(SUM(amount)) AS total
    FROM payments GROUP BY spend_group ORDER BY total DESC
""", conn))
conn.close()

   payments         total
0    141930  1.597050e+09

                  spend_group  payments        total
0                 Procurement     81449  808451670.0
1            Care and support     49272  437394758.0
2  Transfers and non-supplier     11209  351203481.0
